# 07 · XGBoost and bootstrap bagging

Trains XGBoost with early stopping, then "bagging": 10 more XGBoost models per fold, each trained on a random resample (with replacement) of the training patients, averaged.

**Needs:** `features2.parquet` (02), `final_oof.npy` (05). **Writes:** `xgb_oof_single.npy`, `xgb_oof_bag.npy` and their `_test.npy`. **Time:** about 20 minutes.

Result: single XGBoost **0.7507**, bagged **0.7502**. Bagging did not help because shallow boosted trees are already stable. Adding XGBoost to the existing blend also lowered the score a little.

**Setup.** Install the libraries once (uncomment), then set the two paths in the next cell.

In [ ]:
# %pip install -q lightgbm xgboost scikit-learn pandas pyarrow scipy

In [ ]:
import os, sys, json, time, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

# ---- EDIT THESE TWO PATHS --------------------------------------------------
DATA = os.environ.get('KAGGLE_DATA', 'data')   # folder holding the unzipped competition CSVs (train.csv, test.csv, patients.csv, ...)
WORK = os.environ.get('KAGGLE_WORK', 'work')   # folder where features, predictions and submissions are saved (shared by all notebooks)
# -----------------------------------------------------------------------------
Path(WORK).mkdir(parents=True, exist_ok=True)
print('DATA ->', Path(DATA).resolve()); print('WORK ->', Path(WORK).resolve())

In [ ]:
import pandas as pd, numpy as np, xgboost as xgb, json, time
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
D=DATA; B=10
drop=("hrs_","visits_","n_transfers","first_unit","last_unit","icu_","had_ed","ward_after","proc_","n_proc")
X=pd.read_parquet(f'{WORK}/features2.parquet'); tr=pd.read_csv(f'{D}/train.csv'); te=pd.read_csv(f'{D}/test.csv')
for c in [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c]) and c!='encounter_id']: X[c]=X[c].astype('category')
T=X.merge(tr,on='encounter_id'); S=te.merge(X,on='encounter_id',how='left'); y=T.readmit_30d.values
feats=[c for c in T.columns if c not in('encounter_id','readmit_30d') and not c.startswith(drop)]
dS=xgb.DMatrix(S[feats],enable_categorical=True)
P=dict(objective='binary:logistic',eval_metric='auc',tree_method='hist',max_depth=3,eta=0.04,min_child_weight=20,subsample=0.8,colsample_bytree=0.3,
       reg_lambda=20,max_bin=63,max_cat_to_onehot=8,nthread=2)
folds=list(StratifiedKFold(5,shuffle=True,random_state=0).split(T,y))
single=np.zeros(len(T)); bag=np.zeros(len(T)); ts=np.zeros(len(S)); tb=np.zeros(len(S)); rng=np.random.RandomState(7); t0=time.time()
for f,(a,b) in enumerate(folds):
    da=xgb.DMatrix(T.loc[a,feats],y[a],enable_categorical=True); db=xgb.DMatrix(T.loc[b,feats],y[b],enable_categorical=True)
    m=xgb.train({**P,'seed':f},da,4000,evals=[(db,'v')],early_stopping_rounds=200,verbose_eval=False); n=m.best_iteration+1
    single[b]=m.predict(db,iteration_range=(0,n)); ts+=m.predict(dS,iteration_range=(0,n))/5
    for k in range(B):
        idx=rng.choice(a,len(a),replace=True)               # bootstrap resample of training rows
        dk=xgb.DMatrix(T.loc[idx,feats],y[idx],enable_categorical=True)
        mk=xgb.train({**P,'seed':1000*f+k},dk,n)
        bag[b]+=mk.predict(db)/B; tb+=mk.predict(dS)/(5*B)
    print(f'fold {f} rounds={n} single={roc_auc_score(y[b],single[b]):.5f} bagged={roc_auc_score(y[b],bag[b]):.5f} t={time.time()-t0:.0f}s',flush=True)
print('XGB single OOF AUC %.5f'%roc_auc_score(y,single)); print('XGB bagged(B=%d) OOF AUC %.5f'%(B,roc_auc_score(y,bag)))
np.save(f'{WORK}/xgb_oof_single.npy',single); np.save(f'{WORK}/xgb_oof_bag.npy',bag)
np.save(f'{WORK}/xgb_test_single.npy',ts); np.save(f'{WORK}/xgb_test_bag.npy',tb)
print('done')

### 7.1 Does XGBoost help the existing blend? Bootstrap confidence intervals

In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
X = pd.read_parquet(f'{WORK}/features2.parquet', columns=['encounter_id']); tr = pd.read_csv(f'{DATA}/train.csv')
y = X.merge(tr, on='encounter_id').readmit_30d.values
lg = lambda p: np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))
cur = lg(np.load(f'{WORK}/final_oof.npy')); xs = lg(np.load(f'{WORK}/xgb_oof_single.npy')); xb = lg(np.load(f'{WORK}/xgb_oof_bag.npy'))
A = lambda s: roc_auc_score(y, s)
print('current blend %.5f | xgb single %.5f | xgb bagged %.5f' % (A(cur), A(xs), A(xb)))
for w in [.2, .3, .4, .5]:
    print('current + bagged xgb w=%.1f  %.5f' % (w, A((1 - w) * cur + w * xb)), '| + single xgb %.5f' % A((1 - w) * cur + w * xs))
# bootstrap: resample the validation patients to see how big the noise in AUC is
rng = np.random.RandomState(0); n = len(y); cand = {'cur': cur, 'xgb_bag': xb, 'blend30': 0.7 * cur + 0.3 * xb}
res = {k: [] for k in cand}
for _ in range(300):
    i = rng.randint(0, n, n)
    for k, v in cand.items(): res[k].append(roc_auc_score(y[i], v[i]))
res = {k: np.array(v) for k, v in res.items()}
for k, v in res.items(): print(k, 'AUC %.4f  95%% CI [%.4f, %.4f]' % (v.mean(), *np.percentile(v, [2.5, 97.5])))
for k in ['xgb_bag', 'blend30']:
    d = res[k] - res['cur']; print(f'{k} - current: mean {d.mean():+.5f}  95% CI [{np.percentile(d, 2.5):+.5f}, {np.percentile(d, 97.5):+.5f}]')